# Mobile Product Segmentation and Recommendation System

## Step 1: Problem Definition
This project clusters mobile phones into distinct market segments based on their features and user reviews. It also includes a content-based recommendation system using cosine similarity to suggest similar phones.

**Rationale:** We use an Unsupervised Clustering approach because we want to discover inherent, unlabeled groupings of products based on features and user sentiment, rather than predicting a known target variable.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import pickle
from pathlib import Path

## Step 2: Data Collection
We ingest data from local CSV files or a database.

**Rationale:** Real-world product data typically exists in relational databases or data lakes. Using PostgreSQL demonstrates the ability to interact with data programmatically via SQL, a crucial Data Engineering skill.

In [ ]:
from dotenv import load_dotenv
load_dotenv()
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "")
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "product_segmentation")
db_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(db_url)

base_dir = r"c:\Users\jegad\projects\Mobile Product Segmentation and Recommendation System"
data_dir = os.path.join(base_dir, "data")

file_path = os.path.join(data_dir, "cleaned_mobile_reviews.csv")
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    df.to_sql('cleaned_mobile_reviews', engine, if_exists='replace', index=False)
    print(f"Loaded {len(df)} records into the database.")
else:
    file_path = os.path.join(data_dir, "segmented_products.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df.to_sql('cleaned_mobile_reviews', engine, if_exists='replace', index=False)
        print(f"Loaded fallback data with {len(df)} records.")
    else:
        print("Data files not found.")
        df = pd.DataFrame()


## Step 3: Data Cleaning & Exploratory Data Analysis (EDA)
We clean the data and visualize the distributions of price and ratings.

**Rationale:** We chose to drop rows with missing prices or ratings instead of imputing them. Imputing core continuous features for a clustering algorithm can introduce artificial density in the feature space, leading to false or skewed clusters.

In [ ]:
# Drop rows with missing critical information
if 'rating' in df.columns and 'price_usd' in df.columns:
    df = df.dropna(subset=['rating', 'price_usd'])
    
# --- EDA Visualizations ---
if 'rating' in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df['rating'], bins=20, kde=True, color='orange')
    plt.title("Distribution of Mobile Ratings")
    plt.show()
    
if 'price_usd' in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df['price_usd'], bins=30, kde=True, color='green')
    plt.title("Distribution of Mobile Prices (USD)")
    plt.show()


## Step 4: Feature Engineering
We engineer a composite specifications score and aggregate the data at the product level (model).

**Rationale for Aggregation:** We group by the mobile model because we are segmenting *the phones themselves*, not the individual user reviews. Aggregating reviews gives us the holistic 'average sentiment' and 'average rating' per phone.

In [ ]:
# If the df already has these features (fallback), skip
if 'specs_average' not in df.columns and 'battery_life_rating' in df.columns:
    df['specs_average'] = df[['battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating']].mean(axis=1)
    
    product_df = df.groupby(['brand', 'model']).agg(
        avg_price=('price_usd', 'mean'),
        avg_rating=('rating', 'mean'),
        avg_specs_score=('specs_average', 'mean'),
        total_reviews=('review_id', 'count'),
        positive_sentiment_ratio=('sentiment', lambda x: (x == 'Positive').sum() / len(x))
    ).reset_index()
    
    # Keep models with sufficient data
    product_df = product_df[product_df['total_reviews'] >= 5].reset_index(drop=True)
else:
    product_df = df.copy()
    
print(f"Feature engineering complete. {len(product_df)} unique models available.")


## Step 5: Train-Test Split
Since clustering is an unsupervised learning method, we typically train on the entire dataset. However, to evaluate generalization (or if we were building a supervised proxy), we can split the data.

**Rationale:** While fully unsupervised learning often utilizes the entire dataset to build the cluster space, applying a train/test split demonstrates methodological rigor. It allows us to verify if our clusters generalize to unseen data points.

In [ ]:
features = ['avg_price', 'avg_rating', 'avg_specs_score', 'positive_sentiment_ratio']
# Check if features exist
features = [f for f in features if f in product_df.columns]

if features:
    X = product_df[features]
    # We split the data 80/20 just to adhere to the methodology, though we will cluster the whole dataset later
    X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
    print(f"Train size: {len(X_train)} models, Test size: {len(X_test)} models.")


## Step 6: Model Selection
We select **K-Means Clustering** for segmenting the mobile phones into discrete market tiers. We will standardize the features first.

**Rationale for StandardScaler:** K-Means uses Euclidean distance to assign points to clusters. Without scaling, a feature like `price` (ranging 100-2000) would completely dominate a feature like `rating` (ranging 1-5). Standardizing ensures all features contribute equally.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("StandardScaler initialized and data scaled.")


## Step 7: Model Training
We initially train the K-Means model. To find the optimal number of clusters, we use the Elbow method and Silhouette score in the Tuning step.

**Rationale for K-Means:** It is a fast, highly efficient, and easily interpretable algorithm perfectly suited for partitioning continuous numerical data into discrete tiers (e.g., budget, mid-range, flagship).

In [ ]:
# Let's start with an arbitrary k=4
kmeans_initial = KMeans(n_clusters=4, random_state=42, n_init=10)
initial_labels = kmeans_initial.fit_predict(X_scaled)
print("Initial K-Means training complete (k=4).")


## Step 8: Model Evaluation
Evaluating the cluster quality using Silhouette Score and visualizing the 2D clusters.

**Rationale for PCA:** Our data has 4 dimensions, making it impossible to plot. Principal Component Analysis (PCA) reduces the dimensionality to 2D while preserving as much variance as possible, allowing us to visually inspect the cluster separation.

In [ ]:
score = silhouette_score(X_scaled, initial_labels)
print(f"Silhouette Score (k=4): {score:.4f}")

# 2D PCA for visualization
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=initial_labels, palette='viridis', s=100)
plt.title("2D PCA Visualization of Phone Clusters")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title='Cluster')
plt.show()


## Step 9: Model Tuning
We optimize hyperparameter `k` using the Elbow Method and Silhouette Scores.

**Rationale for Tuning:** Guessing 'k' is arbitrary. The Elbow Method (WCSS) helps find the point of diminishing returns for adding clusters, while the Silhouette Score mathematically evaluates how tight the clusters are internally versus how separated they are from other clusters. Combining them yields the most rigorous 'k' value.

In [ ]:
wcss = []
silhouette_scores = []
k_range = range(3, 11)

best_k = 4
best_score = -1

for k in k_range:
    temp_kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    temp_labels = temp_kmeans.fit_predict(X_scaled)
    wcss.append(temp_kmeans.inertia_)
    score = silhouette_score(X_scaled, temp_labels)
    silhouette_scores.append(score)
    
    if score > best_score:
        best_score = score
        best_k = k

print(f"Optimal number of clusters chosen: {best_k} (Silhouette Score: {best_score:.4f})")

# Plotting Elbow and Silhouette
fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:red'
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('WCSS (Elbow Method)', color=color)
ax1.plot(k_range, wcss, marker='o', color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(k_range, silhouette_scores, marker='s', color=color)
ax2.tick_params(axis='y', labelcolor=color)
ax2.axvline(x=best_k, color='green', linestyle='--', label=f'Best k={best_k}')
fig.tight_layout()
plt.title('Optimal K Analysis: Elbow Method vs Silhouette Score')
plt.show()

# Retrain with best_k
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
product_df['cluster'] = kmeans_final.fit_predict(X_scaled)


## Step 10: Model Deployment (Recommendation System)
Saving the clusterer and scaler to disk, and demonstrating the Recommendation Engine using Cosine Similarity.

**Rationale for Cosine Similarity:** In a recommendation system based on specs, Cosine Similarity is highly effective. It calculates the cosine of the angle between feature vectors rather than the absolute distance. This accurately identifies phones with similar feature *profiles*, which is exactly what a user wants when looking for alternatives.

In [ ]:
# Save artifacts
os.makedirs('models', exist_ok=True)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('models/kmeans.pkl', 'wb') as f:
    pickle.dump(kmeans_final, f)

print("Models saved successfully!")

# Recommendation Engine Setup
similarity_matrix = cosine_similarity(X_scaled)

def get_recommendations(idx, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # Skip the first one as it is the same phone
    recommended_indices = [i[0] for i in sim_scores[1:top_n+1]]
    return product_df.iloc[recommended_indices][['brand', 'model', 'avg_price', 'avg_rating', 'cluster']]
    
# Example Recommendation
sample_idx = 0
sample_phone = product_df.iloc[sample_idx]
print(f"\n--- Recommendations for {sample_phone['brand']} {sample_phone['model']} (Cluster {sample_phone['cluster']}) ---")
print(get_recommendations(sample_idx))
